# Bibliotecas

In [34]:
import pandas as pd
import numpy as np
import glob
import os

## CARREGAMENTO, LIMPEZA E TRATAMENTO DOS ARQUIVOS INDIVIDUALMENTE

In [35]:
# Realizar esse processo para cada dataset da pasta uploads
arquivo_csv = ('uploads/processos_1.csv')
df = pd.read_csv(arquivo_csv, sep='#', encoding='utf-8-sig')
print(df.shape)

C:\Users\jcpsrodrigues\AppData\Local\Temp\ipykernel_14320\1877301111.py:3: DtypeWarning: Columns (5,19,20,24,25,41,43,54,56,60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivo_csv, sep='#', encoding='utf-8-sig')


(1851754, 64)


## VALIDAÇÃO RÁPIDA

In [ ]:
pendentes = df['data_baixa'].isna().sum()
print(f"Pendentes: {pendentes}")

Quantidade de valores nulos em 'data_baixa' (pendentes): 1327563


In [ ]:
baixados = df['data_baixa'].notna().sum()
print(f"Baixados: {baixados}")

Quantidade de valores não-nulos em 'data_baixa' (baixados): 524191


In [38]:
tx_cong_global = (100 * pendentes/(pendentes+baixados)).round(2)
print(f"Taxa de Congestionamento Global: {tx_cong_global} %")

Taxa de Congestionamento Global: 71.69 %


In [39]:
# Verificando se na coluna data_baixa e data_distribuicao tem o ANO e quantos são:
# Converte a coluna para o tipo datetime
ANO = 2021
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce') # errors='coerce' transforma erros em NaT (nulo)
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce') 

print("="*60)
print("--- VALIDAÇÃO DE DATA_BAIXA ---")
# Faz a verificação
temp = (df['data_baixa'].dt.year == ANO).any()
print(temp)

# Se quiser saber QUANTOS registros existem com o ano 2019:
qtd = (df['data_baixa'].dt.year == ANO).sum()
print(f"Total de registros de data_baixa em {ANO}: {qtd}")

print("="*60)
print("--- VALIDAÇÃO DE DATA_DISTRIBUICAO ---")
# Faz a verificação
temp2 = (df['data_distribuicao'].dt.year == ANO).any()
print(temp2)

# Se quiser saber QUANTOS registros existem com o ano 2019:
qtd2 = (df['data_distribuicao'].dt.year == ANO).sum()
print(f"Total de registros de data_distribuicao em {ANO}: {qtd2}")

--- VALIDAÇÃO DE DATA_BAIXA ---
True
Total de registros de data_baixa em 2021: 11
--- VALIDAÇÃO DE DATA_DISTRIBUICAO ---
True
Total de registros de data_distribuicao em 2021: 208


## LIMPEZA E TRATAMENTO DAS DATAS DO ARQUIVO_CSV

In [41]:
# Limpeza e tratamento das datas do arquivo_csv
# 1. Converter as colunas de data para datetime
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

# 2. Verificar quantas datas inválidas temos
print(f"\nDatas inválidas na conversão:")
print(f"- data_distribuicao: {df['data_distribuicao'].isna().sum()} valores NaT")
print(f"- data_baixa: {df['data_baixa'].isna().sum()} valores NaT")

# 3. Extrair o ano de cada data
df['ano_distribuicao'] = df['data_distribuicao'].dt.year
df['ano_baixa'] = df['data_baixa'].dt.year

# 4. Fazer a limpeza: manter apenas registros em que:
#    - ano_distribuicao <= 2020 (ou é NaN)
#    - ano_baixa <= 2020 (ou é NaN)
ANO = 2020
df_limpo = df[
    (
        (df['ano_distribuicao'] <= ANO)
    ) & 
    (
        (df['ano_baixa'] <= ANO) | 
        (df['ano_baixa'].isna())
    )
].copy()

print(f"\nShape após limpeza: {df_limpo.shape}")
print(f"Registros removidos: {len(df) - len(df_limpo)}")

# 5. Verificação detalhada
print(f"\nVerificação dos anos nas datas:")
print(f"Anos únicos em data_distribuicao: {sorted(df_limpo['ano_distribuicao'].dropna().unique().astype(int))}")
print(f"Anos únicos em data_baixa: {sorted(df_limpo['ano_baixa'].dropna().unique().astype(int))}")

# 6. Mostrar exemplos de registros removidos (se houver)
if len(df) > len(df_limpo):
    df_removidos = df[~df.index.isin(df_limpo.index)]
    print(f"\nExemplo de registros removidos:")
    print(df_removidos[['data_distribuicao', 'data_baixa', 'ano_distribuicao', 'ano_baixa']].head())
    
    # Contar motivo da remoção
    print(f"\nMotivo da remoção:")
    # Distribuição > 2020
    ANO = 2020
    dist_maior_ANO = df[df['ano_distribuicao'] > ANO]
    dist_nula = df[df['ano_distribuicao'].isna()]
    baixa_maior_ANO = df[df['ano_baixa'] > ANO]
    
    print(f"- data_distribuicao > {ANO}: {len(dist_maior_ANO)} registros")
    print(f"- data_distribuicao NULA: {len(dist_nula)} registros")
    print(f"- data_baixa > {ANO}: {len(baixa_maior_ANO)} registros")
    
    # Intersecção (ambos > ANO)
    ambos_maior_ANO = df[(df['ano_distribuicao'] > ANO) & (df['ano_baixa'] > ANO)]
    print(f"- ambos > {ANO}: {len(ambos_maior_ANO)} registros")

# 7. Opcional: Remover as colunas auxiliares de ano se não precisar mais
# df_limpo = df_limpo.drop(columns=['ano_distribuicao', 'ano_baixa'])

# 8. Mostrar estatísticas do dataframe limpo
print(f"\n{'='*50}")
print("RESUMO DO DATAFRAME LIMPO:")
print(f"{'='*50}")
print(f"Total de registros: {len(df_limpo)}")
print(f"Total de colunas: {len(df_limpo.columns)}")
print(f"\nTipos de dados:")
print(df_limpo.dtypes)

# 9. Verificar valores nulos nas datas
print(f"\nValores nulos nas datas (após limpeza):")
print(f"- data_distribuicao: {df_limpo['data_distribuicao'].isna().sum()} nulos ({df_limpo['data_distribuicao'].isna().sum()/len(df_limpo)*100:.1f}%)")
print(f"- data_baixa: {df_limpo['data_baixa'].isna().sum()} nulos ({df_limpo['data_baixa'].isna().sum()/len(df_limpo)*100:.1f}%)")


Datas inválidas na conversão:
- data_distribuicao: 0 valores NaT
- data_baixa: 1327563 valores NaT

Shape após limpeza: (1851535, 66)
Registros removidos: 219

Verificação dos anos nas datas:
Anos únicos em data_distribuicao: [np.int64(1858), np.int64(1879), np.int64(1936), np.int64(1941), np.int64(1952), np.int64(1954), np.int64(1956), np.int64(1957), np.int64(1958), np.int64(1960), np.int64(1961), np.int64(1963), np.int64(1964), np.int64(1965), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.i

In [42]:
# Validação da Limpeza do dataframe
df_limpo.info()
df_limpo_distribuicao = df_limpo[(df_limpo['ano_distribuicao'] == 2021)].head(10)
df_limpo_baixa = df_limpo[(df_limpo['ano_baixa'] == 2021)].head(10)

print('###################### data_distribuicao com ano = 2021 ######################')
print(df_limpo_distribuicao)
print('######################### data_baixa com ano = 2021 ##########################')
print(df_limpo_baixa)

<class 'pandas.core.frame.DataFrame'>
Index: 1851535 entries, 0 to 1851753
Data columns (total 66 columns):
 #   Column                           Dtype         
---  ------                           -----         
 0   processo_id                      int64         
 1   comarca                          object        
 2   comarca_id                       int64         
 3   entrancia                        object        
 4   serventia                        object        
 5   codg_serventia_cnj               object        
 6   vara_oficial_id                  int64         
 7   origem                           object        
 8   is_conhecimento                  object        
 9   grupo_cnj_id                     int64         
 10  is_recurso                       object        
 11  processo                         object        
 12  natureza                         object        
 13  codg_natureza                    float64       
 14  fase                             object

## GERAR OS ARQUIVOS `clean_process_*.csv` NA PASTA `dataclean`

In [43]:
output_filename_csv_mes = 'dataclean/clean_process_1.csv'
os.makedirs(os.path.dirname(output_filename_csv_mes), exist_ok=True)
df_limpo.to_csv(output_filename_csv_mes, index=False, sep='#', encoding='utf-8-sig')

# Carregamento por Arquivo CSV Limpo

In [44]:
arquivo_csv = ('dataclean/clean_process_1.csv')
df = pd.read_csv(arquivo_csv, sep='#', encoding='utf-8-sig')
print(df.shape)

C:\Users\jcpsrodrigues\AppData\Local\Temp\ipykernel_14320\16687144.py:2: DtypeWarning: Columns (5,19,20,24,25,41,43,54,56,60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivo_csv, sep='#', encoding='utf-8-sig')


(1851535, 66)


# Validações Rápidas

In [45]:
pendentes = df['data_baixa'].isna().sum()
print(f"Pendentes: {pendentes}")

Pendentes: 1327355


In [46]:
baixados = df['data_baixa'].notna().sum()
print(f"Baixados: {baixados}")

Baixados: 524180


In [47]:
tx_cong_global = (100 * pendentes/(pendentes+baixados)).round(2)
print(f"Taxa de Congestionamento Global: {tx_cong_global} %")

Taxa de Congestionamento Global: 71.69 %


# Limpeza e Tratamento

In [48]:
# Verificar o nome correto das colunas (pode haver diferenças de acentuação ou espaços)
colunas = df.columns.tolist()

# Encontrar as colunas de data corretamente
coluna_serventia = [col for col in colunas if 'serventia' in col.lower()][0]
coluna_distribuicao = [col for col in colunas if 'data_distribuicao' in col.lower()][0]
coluna_baixa = [col for col in colunas if 'data_baixa' in col.lower()][0]
coluna_area_acao = [col for col in colunas if 'nome_area_acao' in col.lower()][0]
coluna_processo_id = [col for col in colunas if 'processo_id' in col.lower()][0]
coluna_comarca = [col for col in colunas if 'comarca' in col.lower()][0]

# Renomear colunas para garantir consistência
df = df.rename(columns={
coluna_distribuicao: 'data_distribuicao',
coluna_baixa: 'data_baixa',
coluna_area_acao: 'nome_area_acao',
coluna_processo_id: 'processo_id',
coluna_comarca: 'comarca',
coluna_serventia: 'serventia'
})

# ==========================================================================================================================================
# ANÁLISE MENSAL
# ==========================================================================================================================================

In [149]:
# Análise Mensal
# Gerando o Dataframe com a Taxa de Congestionamento Mensal:
# 0) Garantir datetime
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

keys = ['comarca', 'serventia'] # Altere aqui para fazer os agrupamentos

# 1) Cópia 
df_f = df.copy()

# 2.1) Referência mensal
df_f['mes_dist'] = df_f['data_distribuicao'].dt.to_period('M')
df_f['mes_baixa'] = df_f['data_baixa'].dt.to_period('M')

# 3.1) Fluxos mensais
# Distribuídos por mês (fluxo)
dist_mes_base = (
    df_f.groupby(keys + ['mes_dist'])
        .size()
        .rename('Distribuidos_fluxo_mensal')
        .reset_index()
        .rename(columns={'mes_dist': 'mes_ref'})
)

# Baixados por mês (fluxo)
baix_mes_base = (
    df_f.dropna(subset=['data_baixa'])
        .groupby(keys + ['mes_baixa'])
        .size()
        .rename('Baixados_fluxo_mensal')
        .reset_index()
        .rename(columns={'mes_baixa': 'mes_ref'})
)

# Calcular valores acumulados
# Distribuídos acumulados
dist_acum = dist_mes_base.copy()
dist_acum = dist_acum.sort_values(['comarca', 'serventia', 'mes_ref'])
dist_acum['Distribuidos_acum_mes'] = dist_acum.groupby(keys)['Distribuidos_fluxo_mensal'].cumsum()

# Baixados acumulados
baix_acum = baix_mes_base.copy()
baix_acum = baix_acum.sort_values(['comarca', 'serventia', 'mes_ref'])
baix_acum['Baixados_acum_mes'] = baix_acum.groupby(keys)['Baixados_fluxo_mensal'].cumsum()

# Pendentes acumulados = Distribuídos acumulados - Baixados acumulados
pend_acum = dist_acum[keys + ['mes_ref', 'Distribuidos_acum_mes']].copy()
pend_acum = pend_acum.merge(
    baix_acum[keys + ['mes_ref', 'Baixados_acum_mes']], 
    on=keys + ['mes_ref'], 
    how='left'
)
pend_acum['Baixados_acum_mes'] = pend_acum['Baixados_acum_mes'].fillna(0).astype(int)
pend_acum['Pendentes_acum_mes'] = pend_acum['Distribuidos_acum_mes'] - pend_acum['Baixados_acum_mes']

# 6.1) Dataframe mensal
df_global_mensal = dist_mes_base[['comarca', 'serventia', 'mes_ref', 'Distribuidos_fluxo_mensal']].copy()

# Adicionar Baixados_fluxo_mensal
df_global_mensal = df_global_mensal.merge(
    baix_mes_base[['comarca', 'serventia', 'mes_ref', 'Baixados_fluxo_mensal']],
    on=['comarca', 'serventia', 'mes_ref'],
    how='left'
)

# Adicionar Pendentes_acum_mes
df_global_mensal = df_global_mensal.merge(
    pend_acum[['comarca', 'serventia', 'mes_ref', 'Pendentes_acum_mes']],
    on=['comarca', 'serventia', 'mes_ref'],
    how='left'
)

# Adicionar Distribuidos_acum_mes (apenas para cálculo, depois removemos)
df_global_mensal = df_global_mensal.merge(
    dist_acum[['comarca', 'serventia', 'mes_ref', 'Distribuidos_acum_mes']],
    on=['comarca', 'serventia', 'mes_ref'],
    how='left'
)

# Preencher NaN com 0 e Converter para int
df_global_mensal['Baixados_fluxo_mensal'] = df_global_mensal['Baixados_fluxo_mensal'].fillna(0).astype(int)
df_global_mensal['Pendentes_acum_mes'] = df_global_mensal['Pendentes_acum_mes'].fillna(0).astype(int)
df_global_mensal['Distribuidos_fluxo_mensal'] = df_global_mensal['Distribuidos_fluxo_mensal'].astype(int)
df_global_mensal['Distribuidos_acum_mes'] = df_global_mensal['Distribuidos_acum_mes'].astype(int)

# Renomear colunas
df_global_mensal = df_global_mensal.rename(columns={
    'Distribuidos_fluxo_mensal': 'Distribuidos_mes',
    'Baixados_fluxo_mensal': 'Baixados_mes',
    'Pendentes_acum_mes': 'Pendentes_mes'
})

# Calcular taxa mensal CORRETA: Pendentes_mes / Distribuidos_acum_mes
df_global_mensal['Taxa_Cong_mensal (%)'] = np.where(
    df_global_mensal['Distribuidos_acum_mes'] > 0,
    (df_global_mensal['Pendentes_mes'] / df_global_mensal['Distribuidos_acum_mes']) * 100,
    0
).round(2)

# Remover a coluna auxiliar Distribuidos_acum_mes
df_global_mensal = df_global_mensal.drop('Distribuidos_acum_mes', axis=1)

# Ordenar por comarca, serventia e mês
df_global_mensal = df_global_mensal.sort_values(['comarca', 'serventia', 'mes_ref'])

In [ ]:
# Filtrando todos os MESES de cada ano para exração MENSAL
ano_alvo = 2025
df_mes = df_global_mensal[df_global_mensal['mes_ref'].dt.year == ano_alvo]
df_mes.head(12)

,comarca,serventia,mes_ref,Distribuidos_mes,Baixados_mes,Pendentes_mes,Taxa_Cong_mensal (%)
208,ABADIÂNIA,Vara Judicial,2025-01,142,89,3166,97.27
209,ABADIÂNIA,Vara Judicial,2025-02,148,231,3083,90.60
210,ABADIÂNIA,Vara Judicial,2025-03,129,207,3005,85.08
211,ABADIÂNIA,Vara Judicial,2025-04,170,155,3020,81.58
212,ABADIÂNIA,Vara Judicial,2025-05,188,256,2952,75.89


In [151]:
# Grava cada arquivo MENSAL na pasta results
ano_alvo = 2025
df_mes = df_global_mensal[df_global_mensal['mes_ref'].dt.year == ano_alvo]
os.makedirs("results", exist_ok=True)
df_mes.to_csv(f"results/txmes_{ano_alvo}.csv", sep=",", index=False, encoding="utf-8-sig")

In [152]:
# Concatena os arquivos MENSAIS da pasta results
# Listar os arquivos CSV na pasta 'results'
arquivo_csv = glob.glob('results/txmes_*.csv')
# Carregar os arquivos CSV e concatenar em um único DataFrame
dfs = []
for arquivo in arquivo_csv:  # lista/iterável com os caminhos tipo 'txmes_2020.csv', 'txmes_2021.csv', ...
    df_ano = pd.read_csv(arquivo, sep=',', encoding='utf-8')
    dfs.append(df_ano)

df_results_mensal = pd.concat(dfs, ignore_index=True)

print(df_results_mensal.shape)

(23448, 7)


In [155]:
# Validação Rápida do arquivo MENSAL concatenado
df_results_mensal.head()

,comarca,serventia,mes_ref,Distribuidos_mes,Baixados_mes,Pendentes_mes,Taxa_Cong_mensal (%)
0,ABADIÂNIA,Vara Judicial,2020-01,120,189,3669,95.10
1,ABADIÂNIA,Vara Judicial,2020-02,97,72,3694,93.40
2,ABADIÂNIA,Vara Judicial,2020-03,215,177,3732,89.50
3,ABADIÂNIA,Vara Judicial,2020-04,49,76,3705,87.82
4,ABADIÂNIA,Vara Judicial,2020-05,101,88,3718,86.06


In [157]:
filtro = (
    (df_results_mensal['comarca'] == 'ABADIÂNIA') & 
    (df_results_mensal['serventia'] == 'Vara Judicial') #&
    #(df_results_mensal['mes_ref'].astype(str).str.contains('2021'))
)
df_results_mensal[filtro].head()

,comarca,serventia,mes_ref,Distribuidos_mes,Baixados_mes,Pendentes_mes,Taxa_Cong_mensal (%)
0,ABADIÂNIA,Vara Judicial,2020-01,120,189,3669,95.10
1,ABADIÂNIA,Vara Judicial,2020-02,97,72,3694,93.40
2,ABADIÂNIA,Vara Judicial,2020-03,215,177,3732,89.50
3,ABADIÂNIA,Vara Judicial,2020-04,49,76,3705,87.82
4,ABADIÂNIA,Vara Judicial,2020-05,101,88,3718,86.06


In [158]:
# Grava o arquivo MENSAL concatenado, no formato JSON, na pasta results_concat
df_results_mensal.to_json("results_concat/tx_cong_mensal.json", orient="records", force_ascii=False)